# exp010 — Dev humano de informes

Muestra 200 informes de fold 0, los traduce sin etiquetarlos, prepara los libros de anotación y consolida el dev después de la adjudicación. Los datos y hojas viven siempre bajo el data root.

## 0. Parámetros (única celda con valores)

In [ ]:
EXP = "exp010_dev_informes"
PADRE = "exp001"
SEED = 42
SPLIT = "split_v1_seed42"
SMOKE = True
N_DEV = 200
N_OVERLAP = 40
ANNOTATORS = ["alejandra", "josejavier"]
MIN_POS = 8
OPENAI_MODEL = "gpt-5.4-mini"

TRAIN_CSV_REL = "raw/train.csv"
STUDY_META_REL = "cache/meta_v2/study_meta_v2.csv"
V1_LABELS_REL = "labels/report_labels_v1.csv"
SAMPLE_CSV_REL = "labels/exp010_muestra.csv"
TRANSLATION_CACHE_REL = "labels/exp010_cache"
ANNOTATION_DIR_REL = "labels/exp010_anotacion"
FINAL_CSV_REL = "labels/dev_informes_v1.csv"

## 1. Setup

In [ ]:
import json
import os
import random
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import openpyxl
import pandas as pd
from IPython.display import display
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.worksheet.datavalidation import DataValidation
from sklearn.metrics import cohen_kappa_score

HERE = Path.cwd()
REPO = next(p for p in [HERE, *HERE.parents] if (p / "src" / "knee").exists())
sys.path.insert(0, str(REPO))

def load_secret_env(path):
    if not path.exists():
        return
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        if key and key not in os.environ:
            os.environ[key] = value.strip().strip(chr(34)).strip(chr(39))

load_secret_env(REPO / ".secrets.env")
from src.knee.config import DATA_ROOT, LABELS
from src.knee.experiments import git_info, guardar_metricas
from src.knee.splits import GOLD_FOLD, load_split

TRAIN_CSV = DATA_ROOT / TRAIN_CSV_REL
STUDY_META_CSV = DATA_ROOT / STUDY_META_REL
V1_LABELS_CSV = DATA_ROOT / V1_LABELS_REL
SAMPLE_CSV = DATA_ROOT / SAMPLE_CSV_REL
TRANSLATION_CACHE = DATA_ROOT / TRANSLATION_CACHE_REL
ANNOTATION_DIR = DATA_ROOT / ANNOTATION_DIR_REL
FINAL_CSV = DATA_ROOT / FINAL_CSV_REL
RESULTS = REPO / "results" / EXP

assert (REPO / "experiments" / EXP / "SPEC.md").exists(), "Primero confirma la SPEC"
for folder in [SAMPLE_CSV.parent, TRANSLATION_CACHE, ANNOTATION_DIR, RESULTS]:
    folder.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
AUTHOR = os.environ.get("RSNA_USER", "rodrigo")
GIT = git_info(REPO)
print(f"{EXP} · data root {DATA_ROOT} · git {GIT['commit']} ({GIT['branch']})")

## 2. Muestreo estratificado

In [ ]:
split = load_split(SPLIT)
assert GOLD_FOLD == -1
fold0 = split.loc[split.fold.eq(0), ["StudyInstanceUID", "fold"]].copy()
assert len(fold0) >= N_DEV and not fold0.StudyInstanceUID.isin(split.loc[split.fold.lt(0), "StudyInstanceUID"]).any()

train = pd.read_csv(TRAIN_CSV, encoding="utf-8", dtype={"StudyInstanceUID": str})
meta = pd.read_csv(STUDY_META_CSV, dtype={"study": str})[["study", "lang"]]
meta = meta.rename(columns={"study": "StudyInstanceUID", "lang": "idioma"})
pool = fold0.merge(train[["StudyInstanceUID", "Report"]], on="StudyInstanceUID", how="inner")
pool = pool.merge(meta, on="StudyInstanceUID", how="left")
pool["idioma"] = pool.idioma.fillna("unknown").astype(str)
assert len(pool) == len(fold0) and pool.StudyInstanceUID.is_unique

excluded_label_files = {SAMPLE_CSV.resolve(), FINAL_CSV.resolve()}
labeler_files = [V1_LABELS_CSV]
for path in sorted((DATA_ROOT / "labels").rglob("*.csv")):
    if path.resolve() not in excluded_label_files and not path.name.startswith("dev_informes") and path not in labeler_files:
        labeler_files.append(path)

union = pd.DataFrame(False, index=pool.StudyInstanceUID, columns=LABELS)
used_labelers = []
for path in labeler_files:
    if not path.exists():
        continue
    candidate = pd.read_csv(path, dtype={"StudyInstanceUID": str})
    if not {"StudyInstanceUID", *LABELS}.issubset(candidate.columns):
        continue
    positive = candidate[["StudyInstanceUID", *LABELS]].copy()
    positive[LABELS] = positive[LABELS].apply(pd.to_numeric, errors="coerce").eq(1)
    positive = positive.groupby("StudyInstanceUID")[LABELS].max()
    common = union.index.intersection(positive.index)
    union.loc[common] = union.loc[common] | positive.loc[common]
    used_labelers.append(path.name)
assert V1_LABELS_CSV.name in used_labelers, f"Falta {V1_LABELS_CSV}"
for label in LABELS:
    pool[f"_union_{label}"] = pool.StudyInstanceUID.map(union[label]).fillna(False).astype(bool)

def language_quotas(counts, target):
    counts = counts.sort_index()
    minimum = counts.clip(upper=10).astype(int)
    if int(minimum.sum()) > target:
        raise ValueError("N_DEV no alcanza para el mínimo de idiomas")
    raw = counts / counts.sum() * target
    quotas = np.floor(raw).astype(int).clip(lower=minimum, upper=counts)
    while int(quotas.sum()) < target:
        candidates = counts.index[quotas < counts]
        language = sorted(candidates, key=lambda item: (-(raw[item] - quotas[item]), str(item)))[0]
        quotas[language] += 1
    while int(quotas.sum()) > target:
        candidates = counts.index[quotas > minimum]
        language = sorted(candidates, key=lambda item: (-(quotas[item] - raw[item]), str(item)))[0]
        quotas[language] -= 1
    return quotas.astype(int)

language_counts = pool.idioma.value_counts()
quotas = language_quotas(language_counts, N_DEV)
rng = np.random.default_rng(SEED)
selected = set()
for language, quota in quotas.items():
    choices = pool.index[pool.idioma.eq(language)].to_numpy()
    selected.update(rng.choice(choices, size=int(quota), replace=False).tolist())

union_columns = [f"_union_{label}" for label in LABELS]
def selected_positive_counts(indexes):
    return pool.loc[list(indexes), union_columns].sum().rename(lambda name: name.removeprefix("_union_")).astype(int)

for _ in range(len(pool)):
    current = selected_positive_counts(selected)
    deficits = {label: max(0, MIN_POS - int(current[label])) for label in LABELS}
    if not any(deficits.values()):
        break
    current_gap = sum(deficits.values())
    best = None
    for incoming in pool.index.difference(pd.Index(selected)):
        incoming_row = pool.loc[incoming]
        if not any(deficits[label] and incoming_row[f"_union_{label}"] for label in LABELS):
            continue
        removable = [idx for idx in selected if pool.at[idx, "idioma"] == incoming_row.idioma]
        for outgoing in removable:
            new_counts = current.copy()
            for label in LABELS:
                new_counts[label] += int(incoming_row[f"_union_{label}"]) - int(pool.at[outgoing, f"_union_{label}"] )
            gap = sum(max(0, MIN_POS - int(new_counts[label])) for label in LABELS)
            key = (gap, str(pool.at[outgoing, "StudyInstanceUID"]), str(incoming_row.StudyInstanceUID))
            if best is None or key < best[0]:
                best = (key, outgoing, incoming)
    if best is None or best[0][0] >= current_gap:
        missing = [label for label, value in deficits.items() if value]
        raise RuntimeError(f"No se pudo enriquecer la muestra para: {missing}")
    selected.remove(best[1])
    selected.add(best[2])
else:
    raise RuntimeError("El enriquecimiento no convergió")

dev = pool.loc[sorted(selected), ["StudyInstanceUID", "idioma", "Report"]].copy()
dev = dev.rename(columns={"Report": "informe original"}).sort_values("StudyInstanceUID").reset_index(drop=True)
assert len(dev) == N_DEV and dev.StudyInstanceUID.is_unique
assert set(dev.StudyInstanceUID) <= set(fold0.StudyInstanceUID)
dev.to_csv(SAMPLE_CSV, index=False)
positive_counts_union = selected_positive_counts(selected).to_dict()
assert min(positive_counts_union.values()) >= MIN_POS
print("Etiquetadores usados solo para muestrear:", used_labelers)
display(dev.groupby("idioma").size().rename("n").to_frame())
display(pd.Series(positive_counts_union, name="positivos_union").to_frame())

## 3. Traducción GPT (reanudable)

In [ ]:
PRICE_PER_MILLION = {"gpt-5.4-mini": {"input": 0.75, "output": 4.50}}
PRICING_SOURCE = "https://developers.openai.com/api/docs/models/gpt-5.4-mini"
TRANSLATION_INSTRUCTIONS = (
    "Translate the knee MRI radiology report faithfully into English. Preserve every negation, degree, "
    "uncertainty and hedge, including meanings such as 'possible', 'suggestive of' and 'cannot be excluded'. "
    "Do not summarize, interpret, diagnose, classify, or output labels. Return only the English translation."
)

def cache_path(uid):
    return TRANSLATION_CACHE / f"{uid}.json"

def read_cached_translation(uid):
    path = cache_path(uid)
    if not path.exists():
        return None
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return None
    return payload if payload.get("model") == OPENAI_MODEL and payload.get("translation") else None

def estimated_cost(input_tokens, output_tokens):
    price = PRICE_PER_MILLION.get(OPENAI_MODEL)
    if price is None:
        return None
    return (input_tokens * price["input"] + output_tokens * price["output"]) / 1_000_000

targets = dev.head(5) if SMOKE else dev
api_key = os.getenv("OPENAI_API_KEY", "").strip()
if not api_key:
    print("OPENAI_API_KEY no disponible: traducción pendiente; se exportarán celdas vacías.")
else:
    from openai import BadRequestError, OpenAI
    client = OpenAI(api_key=api_key)
    for uid, language, report in targets[["StudyInstanceUID", "idioma", "informe original"]].itertuples(index=False, name=None):
        uid = str(uid)
        if read_cached_translation(uid):
            continue
        request = {
            "model": OPENAI_MODEL,
            "instructions": TRANSLATION_INSTRUCTIONS,
            "input": f"Source language: {language}\n\nREPORT:\n{report}",
            "temperature": 0,
            "store": False,
        }
        try:
            try:
                response = client.responses.create(**request)
            except BadRequestError as exc:
                if "temperature" not in str(exc).lower():
                    raise
                request.pop("temperature")
                response = client.responses.create(**request)
            input_tokens = int(getattr(response.usage, "input_tokens", 0) or 0)
            output_tokens = int(getattr(response.usage, "output_tokens", 0) or 0)
            payload = {
                "StudyInstanceUID": uid,
                "model": OPENAI_MODEL,
                "translation": response.output_text.strip(),
                "input_tokens": input_tokens,
                "output_tokens": output_tokens,
                "estimated_cost_usd": estimated_cost(input_tokens, output_tokens),
                "pricing_source": PRICING_SOURCE,
                "created_at_utc": datetime.now(timezone.utc).isoformat(),
            }
            temporary = cache_path(uid).with_suffix(".tmp")
            temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
            os.replace(temporary, cache_path(uid))
        except Exception as exc:
            print(f"Traducción fallida para {uid}: {type(exc).__name__}: {exc}")

cached = {str(uid): read_cached_translation(str(uid)) for uid in dev.StudyInstanceUID}
dev["traduccion"] = dev.StudyInstanceUID.map(lambda uid: (cached.get(str(uid)) or {}).get("translation", ""))
valid_cache = [payload for payload in cached.values() if payload]
known_costs = [payload.get("estimated_cost_usd") for payload in valid_cache if payload.get("estimated_cost_usd") is not None]
translation_stats = {
    "model": OPENAI_MODEL,
    "n_cached": len(valid_cache),
    "input_tokens": sum(int(payload.get("input_tokens", 0)) for payload in valid_cache),
    "output_tokens": sum(int(payload.get("output_tokens", 0)) for payload in valid_cache),
    "estimated_cost_usd": float(sum(known_costs)) if len(known_costs) == len(valid_cache) else None,
    "pricing_source": PRICING_SOURCE,
}
print(f"Traducciones disponibles: {len(valid_cache)}/{len(dev)} · costo estimado: {translation_stats['estimated_cost_usd']} USD")

## 4. Exportar libros de anotación

In [ ]:
ALLOWED_STATUSES = ["present", "absent", "not_mentioned", "uncertain"]
OFFICIAL_SOURCE = "https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/discussion/733343"
OFFICIAL_DEFINITIONS = {
    "ACL": "Rotura parcial de alto grado o completa del ligamento cruzado anterior: discontinuidad completa o más del 50 % de fibras rotas. Cambio leve de señal, degeneración o engrosamiento sin discontinuidad es negativo.",
    "MCL": "Rotura aguda parcial de alto grado o completa del ligamento colateral medial, con fibras interrumpidas y edema dentro y alrededor del ligamento. Esguinces leves o cambios crónicos son negativos.",
    "Medial Meniscus": "Señal anormal que contacta definitivamente la superficie del menisco medial en al menos dos imágenes, o fragmento truncado, pequeño o desplazado. Degeneración intrameniscal sin llegar a la superficie es negativa.",
    "Lateral Meniscus": "Los mismos criterios de rotura meniscal, aplicados al menisco lateral.",
    "Medial OA": "Área moderada o grande (aprox. 1 cm o más) de pérdida de cartílago de alto grado, mayor del 50 % del grosor, en el compartimento medial, con o sin cambios subcondrales.",
    "Lateral OA": "Los mismos criterios de artrosis, aplicados al compartimento tibiofemoral lateral.",
    "PF OA": "Los mismos criterios de artrosis, aplicados al compartimento patelofemoral.",
    "Effusion": "Cantidad moderada o grande de líquido que distiende la articulación.",
    "Synovitis": "Inflamación y engrosamiento del revestimiento sinovial de la articulación.",
    "Baker's": "Colección líquida moderada o grande en la localización característica posterior de un quiste de Baker o poplíteo.",
    "Contusion": "Contusión ósea: señal tipo edema de médula ósea por impacto, sin una línea de fractura definida.",
    "Fracture": "Fractura aguda con interrupción cortical o línea de fractura.",
}

assert len(ANNOTATORS) == 2 and 0 <= N_OVERLAP <= N_DEV
assert (N_DEV - N_OVERLAP) % len(ANNOTATORS) == 0
assignment_rng = np.random.default_rng(SEED + 101)
all_ids = dev.StudyInstanceUID.to_numpy()
overlap_ids = set(assignment_rng.choice(all_ids, size=N_OVERLAP, replace=False).tolist())
remaining_ids = assignment_rng.permutation([uid for uid in all_ids if uid not in overlap_ids])
chunks = np.array_split(remaining_ids, len(ANNOTATORS))
assignments = {annotator: overlap_ids | set(chunks[index].tolist()) for index, annotator in enumerate(ANNOTATORS)}
expected_per_annotator = N_OVERLAP + (N_DEV - N_OVERLAP) // len(ANNOTATORS)
assert all(len(ids) == expected_per_annotator for ids in assignments.values())

def style_header(sheet):
    for cell in sheet[1]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = PatternFill("solid", fgColor="1F4E78")
        cell.alignment = Alignment(wrap_text=True, vertical="top")

def write_annotation_workbook(path, frame):
    workbook = openpyxl.Workbook()
    sheet = workbook.active
    sheet.title = "anotar"
    headers = ["StudyInstanceUID", "idioma", "informe original", "traduccion", *LABELS, "evidencia / notas"]
    sheet.append(headers)
    for row in frame.itertuples(index=False, name=None):
        sheet.append([row[0], row[1], row[2], row[3], *([None] * len(LABELS)), None])
    style_header(sheet)
    sheet.freeze_panes = "A2"
    sheet.auto_filter.ref = sheet.dimensions
    widths = {"A": 42, "B": 12, "C": 70, "D": 70, "Q": 45}
    for column, width in widths.items():
        sheet.column_dimensions[column].width = width
    for column in range(5, 17):
        sheet.column_dimensions[openpyxl.utils.get_column_letter(column)].width = 20
    for row in sheet.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(wrap_text=True, vertical="top")
    for column in range(5, 17):
        validation = DataValidation(type="list", formula1='"present,absent,not_mentioned,uncertain"', allow_blank=False)
        validation.error = "Elige uno de los cuatro estados permitidos"
        validation.errorTitle = "Valor inválido"
        sheet.add_data_validation(validation)
        letter = openpyxl.utils.get_column_letter(column)
        validation.add(f"{letter}2:{letter}{sheet.max_row}")
    glossary = workbook.create_sheet("glosario")
    glossary.append(["Etiqueta", "Definición oficial (traducción al español)"])
    for label in LABELS:
        glossary.append([label, OFFICIAL_DEFINITIONS[label]])
    glossary.append([])
    glossary.append(["Fuente oficial", OFFICIAL_SOURCE])
    glossary.append(["Regla", "No mencionado → not_mentioned; negado explícitamente → absent; posible/no se descarta/sugestivo → uncertain."])
    glossary.append(["Lateralidad", "Izquierda/derecha es irrelevante: cada estudio corresponde a una sola rodilla."])
    style_header(glossary)
    glossary.freeze_panes = "A2"
    glossary.column_dimensions["A"].width = 24
    glossary.column_dimensions["B"].width = 110
    for row in glossary.iter_rows():
        for cell in row:
            cell.alignment = Alignment(wrap_text=True, vertical="top")
    workbook.save(path)

def sync_translations(path, frame):
    workbook = openpyxl.load_workbook(path)
    sheet = workbook["anotar"]
    expected = set(frame.StudyInstanceUID.astype(str))
    found = {str(sheet.cell(row, 1).value) for row in range(2, sheet.max_row + 1)}
    if found != expected:
        raise ValueError(f"{path.name} no corresponde a la muestra actual")
    translations = frame.set_index("StudyInstanceUID").traduccion.to_dict()
    changed = False
    for row in range(2, sheet.max_row + 1):
        uid = str(sheet.cell(row, 1).value)
        if not sheet.cell(row, 4).value and translations.get(uid):
            sheet.cell(row, 4).value = translations[uid]
            changed = True
    if changed:
        workbook.save(path)

workbook_paths = {}
for index, annotator in enumerate(ANNOTATORS):
    assigned = dev[dev.StudyInstanceUID.isin(assignments[annotator])][["StudyInstanceUID", "idioma", "informe original", "traduccion"]]
    assigned = assigned.iloc[np.random.default_rng(SEED + 1000 + index).permutation(len(assigned))].reset_index(drop=True)
    path = ANNOTATION_DIR / f"exp010_anotacion_{annotator}.xlsx"
    workbook_paths[annotator] = path
    if path.exists():
        sync_translations(path, assigned)
        print(f"Conservado (no se sobrescribe): {path}")
    else:
        write_annotation_workbook(path, assigned)
        print(f"Creado: {path} ({len(assigned)} estudios)")

## 5. Importar, validar, calcular kappa y adjudicar

In [ ]:
def read_annotation(path, annotator):
    if not path.exists():
        print(f"pendiente: falta {path.name}")
        return None
    frame = pd.read_excel(path, sheet_name="anotar", dtype={"StudyInstanceUID": str})
    required = ["StudyInstanceUID", *LABELS, "evidencia / notas"]
    missing_columns = [column for column in required if column not in frame.columns]
    if missing_columns:
        raise ValueError(f"{path.name}: faltan columnas {missing_columns}")
    if frame.StudyInstanceUID.duplicated().any() or set(frame.StudyInstanceUID) != assignments[annotator]:
        raise ValueError(f"{path.name}: estudios duplicados o distintos de la asignación")
    invalid = []
    blank = 0
    for label in LABELS:
        values = frame[label]
        blank += int(values.fillna("").astype(str).str.strip().eq("").sum())
        invalid.extend(values[values.notna() & ~values.isin(ALLOWED_STATUSES)].astype(str).unique().tolist())
    if invalid:
        raise ValueError(f"{path.name}: valores inválidos {sorted(set(invalid))}")
    if blank:
        print(f"pendiente: {path.name} tiene {blank} etiquetas vacías")
        return None
    frame["annotator"] = annotator
    return frame

annotations = {annotator: read_annotation(workbook_paths[annotator], annotator) for annotator in ANNOTATORS}
annotations_ready = all(frame is not None for frame in annotations.values())
kappa_four = {}
kappa_binary = {}
kappa_table = pd.DataFrame()
final_dev = None
adjudication_path = ANNOTATION_DIR / "exp010_adjudicacion.xlsx"

def safe_kappa(left, right, labels):
    if pd.concat([left, right], ignore_index=True).nunique() <= 1:
        return np.nan
    return float(cohen_kappa_score(left, right, labels=labels))

def write_adjudication(path, disagreements, previous):
    workbook = openpyxl.Workbook()
    sheet = workbook.active
    sheet.title = "adjudicar"
    columns = ["StudyInstanceUID", "idioma", "informe original", "traduccion", "label", *ANNOTATORS, "adjudicacion"]
    sheet.append(columns)
    for row in disagreements:
        key = (row["StudyInstanceUID"], row["label"])
        sheet.append([row.get(column, previous.get(key) if column == "adjudicacion" else None) for column in columns])
    style_header(sheet)
    sheet.freeze_panes = "A2"
    sheet.auto_filter.ref = sheet.dimensions
    for column, width in {"A": 42, "B": 12, "C": 70, "D": 70, "E": 22, "F": 18, "G": 18, "H": 20}.items():
        sheet.column_dimensions[column].width = width
    for row in sheet.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(wrap_text=True, vertical="top")
    if sheet.max_row >= 2:
        validation = DataValidation(type="list", formula1='"present,absent,not_mentioned,uncertain"', allow_blank=False)
        sheet.add_data_validation(validation)
        validation.add(f"H2:H{sheet.max_row}")
    workbook.save(path)

if not annotations_ready:
    print("pendiente: completa ambos libros; se omiten kappa, adjudicación y CSV final")
else:
    left = annotations[ANNOTATORS[0]].set_index("StudyInstanceUID")
    right = annotations[ANNOTATORS[1]].set_index("StudyInstanceUID")
    overlap = sorted(left.index.intersection(right.index))
    assert len(overlap) == N_OVERLAP
    for label in LABELS:
        kappa_four[label] = safe_kappa(left.loc[overlap, label], right.loc[overlap, label], ALLOWED_STATUSES)
        left_binary = left.loc[overlap, label].eq("present").astype(int)
        right_binary = right.loc[overlap, label].eq("present").astype(int)
        kappa_binary[label] = safe_kappa(left_binary, right_binary, [0, 1])
    kappa_table = pd.DataFrame({"kappa_4": kappa_four, "kappa_present": kappa_binary})
    kappa_table.loc["macro"] = kappa_table.mean(skipna=True)
    display(kappa_table.round(3))

    dev_lookup = dev.set_index("StudyInstanceUID")
    disagreements = []
    for uid in overlap:
        for label in LABELS:
            if left.at[uid, label] != right.at[uid, label]:
                disagreements.append({
                    "StudyInstanceUID": uid, "idioma": dev_lookup.at[uid, "idioma"],
                    "informe original": dev_lookup.at[uid, "informe original"],
                    "traduccion": dev_lookup.at[uid, "traduccion"], "label": label,
                    ANNOTATORS[0]: left.at[uid, label], ANNOTATORS[1]: right.at[uid, label],
                })

    previous = {}
    if adjudication_path.exists():
        old = pd.read_excel(adjudication_path, sheet_name="adjudicar", dtype={"StudyInstanceUID": str})
        if "adjudicacion" in old.columns:
            invalid = old.loc[old.adjudicacion.notna() & ~old.adjudicacion.isin(ALLOWED_STATUSES), "adjudicacion"]
            if not invalid.empty:
                raise ValueError(f"Adjudicaciones inválidas: {sorted(invalid.astype(str).unique())}")
            previous = {(str(row.StudyInstanceUID), row.label): row.adjudicacion for row in old.itertuples() if pd.notna(row.adjudicacion)}
    write_adjudication(adjudication_path, disagreements, previous)
    print(f"Desacuerdos: {len(disagreements)} · {adjudication_path}")

    disagreement_keys = {(row["StudyInstanceUID"], row["label"]) for row in disagreements}
    if not disagreement_keys.issubset(previous):
        print(f"pendiente: Rodrigo debe adjudicar {len(disagreement_keys - set(previous))} desacuerdos")
    else:
        records = []
        indexed = {annotator: frame.set_index("StudyInstanceUID") for annotator, frame in annotations.items()}
        for uid in dev.StudyInstanceUID:
            record = {"StudyInstanceUID": uid}
            for label in LABELS:
                values = [frame.at[uid, label] for frame in indexed.values() if uid in frame.index]
                if len(values) == 1 or values[0] == values[1]:
                    record[label] = values[0]
                else:
                    record[label] = previous[(uid, label)]
            records.append(record)
        final_dev = pd.DataFrame(records)
        numeric = {"present": 1.0, "absent": 0.0, "not_mentioned": 0.0, "uncertain": np.nan}
        for label in LABELS:
            final_dev[f"{label}_numeric"] = final_dev[label].map(numeric)
        final_dev.to_csv(FINAL_CSV, index=False)
        print(f"Guardado: {FINAL_CSV} ({len(final_dev)} estudios)")

## 6. Guardar `metrics.json`

In [ ]:
def json_number(value):
    return float(value) if value is not None and np.isfinite(value) else None

positive_counts = ({label: int(final_dev[label].eq("present").sum()) for label in LABELS}
                   if final_dev is not None else {label: None for label in LABELS})
metrics = {
    "schema_version": 1,
    "exp": EXP,
    "parent": PADRE,
    "author": AUTHOR,
    "split": SPLIT,
    "split_sha256": split.attrs["sha256"],
    "git": GIT,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "phase": "smoke" if SMOKE else "annotation",
    "status": "complete" if final_dev is not None else "pending",
    "n_sampled": len(dev),
    "n_studies": len(final_dev) if final_dev is not None else 0,
    "languages": {str(key): int(value) for key, value in dev.idioma.value_counts().sort_index().items()},
    "positives_per_label": positive_counts,
    "kappa": {
        "four_categories": {label: json_number(value) for label, value in kappa_four.items()},
        "present_vs_rest": {label: json_number(value) for label, value in kappa_binary.items()},
        "macro_four_categories": json_number(kappa_table.loc["macro", "kappa_4"]) if not kappa_table.empty else None,
        "macro_present_vs_rest": json_number(kappa_table.loc["macro", "kappa_present"]) if not kappa_table.empty else None,
    },
    "translation": translation_stats,
    "model": OPENAI_MODEL,
}
print("Guardado:", guardar_metricas(RESULTS, metrics))

## 7. Verdict

Cuando estén las 200 anotaciones y la adjudicación, copia únicamente los números de `results/exp010_dev_informes/metrics.json` a la sección 3 de `SPEC.md` y actualiza `docs/EXPERIMENTOS.md`. No cambies los criterios después de ver los resultados.